# 🌐 POLARIS v3 — Multi-Agent AI Governance Engine
### *An OpenEnv Environment for Training LLMs on Multi-Agent Negotiation, Theory-of-Mind, and Long-Horizon Planning*

**Built by [Abhishek A S](https://github.com/abhishekascodes) — Solo, Age 17**

---

This notebook demonstrates POLARIS v3, a research-grade RL environment where:
- **5 LLM-powered minister personas** generate proposals and negotiate
- **Theory-of-Mind scoring** rewards agents for predicting vetoes
- **Long-horizon briefings** test multi-step planning across 200+ steps
- **6-component composite reward** prevents reward hacking

In [ ]:
# Install dependencies
!pip install -q torch transformers trl accelerate datasets matplotlib openai fastapi uvicorn pydantic pyyaml

In [ ]:
# Clone the repository
!git clone https://github.com/abhishekascodes/POLARIS-V3.git
import os
os.chdir('POLARIS-V3')
print('✅ Repository cloned')

## 1. Environment Overview
POLARIS simulates a nation with 21 interconnected metrics (GDP, pollution, satisfaction, etc.). The agent must govern by choosing policy actions while navigating a council of 5 ministers who can veto decisions.

In [ ]:
import sys
sys.path.insert(0, '.')
from server.policy_environment import PolicyEnvironment
from server.config import TASK_CONFIGS, VALID_ACTIONS

print('🌐 POLARIS v3 — Available Tasks:')
print('=' * 60)
for tid, cfg in TASK_CONFIGS.items():
    print(f"  {tid}")
    print(f"    {cfg['description']}")
    print(f"    Max steps: {cfg['max_steps']} | Negotiation: {cfg.get('enable_negotiation', False)}")
    print()

print(f'\n📋 Available Actions ({len(VALID_ACTIONS)}):')
for a in sorted(VALID_ACTIONS):
    print(f'  • {a}')

## 2. Running an Episode
Let's run the full negotiation arena — 5 ministers, proposals, coalitions, and veto predictions.

In [ ]:
import random

env = PolicyEnvironment()
obs = env.reset(seed=42, task_id='negotiation_arena')

print('🏛️ NEGOTIATION ARENA — Episode Start')
print('=' * 60)

total_reward = 0
actions = sorted(list(VALID_ACTIONS))

for step in range(30):
    if obs.done:
        print(f'\n💥 Episode ended at step {step}')
        break
    
    # Smart heuristic policy
    meta = obs.metadata
    sat = meta.get('public_satisfaction', 50)
    poll = meta.get('pollution_index', 100)
    gdp = meta.get('gdp_index', 100)
    
    if sat < 25: action = 'increase_welfare'
    elif poll > 200: action = 'enforce_emission_limits'
    elif gdp < 50: action = 'stimulate_economy'
    else: action = actions[step % len(actions)]
    
    action_data = {
        'action': action,
        'reasoning': f'Step {step}: responding to state',
        'coalition_target': ['Chancellor Voss'],
        'veto_prediction': [],
        'stance': 'cooperative'
    }
    
    obs = env.step(action_data)
    total_reward += obs.reward
    
    outcome = meta.get('negotiation_outcome', {})
    if step % 5 == 0:
        print(f'\nStep {step}: {action} → reward={obs.reward:.2f}')
        print(f'  GDP={gdp:.0f} | Pollution={poll:.0f} | Satisfaction={sat:.0f}')
        narrative = meta.get('negotiation_narrative', '')
        if narrative:
            print(f'  Council: {narrative[:150]}...')

collapsed = obs.metadata.get('collapsed', False)
print(f'\n{"=" * 60}')
print(f'Total Reward: {total_reward:.2f}')
print(f'Status: {"COLLAPSED 💥" if collapsed else "SURVIVED ✅"}')

## 3. Benchmark Results — Llama 3.3 70B vs POLARIS
I benchmarked Llama 3.3 70B (via Groq API) against all 6 tasks. The results prove the environment genuinely challenges frontier models.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Real benchmark data from Llama 3.3 70B
tasks = ['env_recovery\n(Easy)', 'balanced\n_economy', 'sust_gov\n(Hard)', 'sust_gov\n_extreme', 'multi_agent\n_council', 'negotiation\n_arena']
llama_scores = [0.9625, 0.1507, 0.1734, 0.2814, 0.2013, 0.2260]
random_scores = [0.7622, 0.1544, 0.1715, 0.2932, 0.2181, 0.2198]
tom_accuracy = [None, None, 4, 0, 0, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('POLARIS v3 — Llama 3.3 70B Benchmark', fontsize=14, fontweight='bold')

# Task scores
ax = axes[0]
x = np.arange(len(tasks))
ax.bar(x - 0.2, llama_scores, 0.35, label='Llama 70B', color='#4f46e5', alpha=0.9)
ax.bar(x + 0.2, random_scores, 0.35, label='Random', color='#a1a1aa', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(tasks, fontsize=8)
ax.set_ylabel('Score'); ax.set_title('Task Scores: LLM vs Random')
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.3, color='red', linestyle='--', alpha=0.3, label='Collapse threshold')

# ToM accuracy
ax = axes[1]
tom_tasks = ['sust_gov\n(Hard)', 'sust_gov\n_extreme', 'multi_agent\n_council', 'negotiation\n_arena']
tom_vals = [4, 0, 0, 0]
colors = ['#d97706' if v > 0 else '#e11d48' for v in tom_vals]
ax.bar(tom_tasks, tom_vals, color=colors, width=0.5)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Theory-of-Mind: Veto Prediction Accuracy')
ax.set_ylim(0, 100)
ax.text(0.5, 0.7, 'Llama 70B: 0-4%\nCannot predict vetoes', transform=ax.transAxes,
        ha='center', fontsize=14, fontweight='bold', color='#e11d48')

plt.tight_layout()
plt.savefig('benchmark_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n🔥 Key Finding: Llama 70B scores 0.96 on easy tasks but COLLAPSES to 0.22 on negotiation')
print('🧠 Theory-of-Mind accuracy: 0-4% — frontier LLMs cannot predict minister vetoes')

## 4. GRPO Training Pipeline
The training script uses TRL's GRPO algorithm with a 6-component reward function to train agents on multi-agent governance.

In [ ]:
# Show the reward components
print('💰 POLARIS v3 — 6-Component Reward Function')
print('=' * 55)
components = [
    ('Base Governance',    '~40%', 'Multi-metric improvement'),
    ('Pareto Optimality',  '~15%', 'Balanced improvement across all dims'),
    ('Theory-of-Mind',     '~15%', 'Correct veto predictions (+0.15)'),
    ('Cooperation',        '~10%', 'Coalition formation bonus'),
    ('Briefing Compliance','~10%', 'Acting on timed intelligence'),
    ('Oscillation Penalty','~10%', 'Penalizes flip-flopping'),
]
for name, weight, desc in components:
    print(f'  {name:>22s} [{weight}] — {desc}')

print('\n📊 Curriculum Escalation (from GRPO training):')
print('-' * 55)
curriculum = [
    ('Easy',    0.0, 39.78, '3/3', '0%'),
    ('Medium',  0.3, 23.96, '0/3', '0%'),
    ('Hard',    0.6, 19.59, '0/3', '0%'),
    ('Extreme', 1.0, 13.88, '0/3', '0%'),
]
print(f'{"Level":>8s}  {"Chaos":>5s}  {"Reward":>7s}  {"Survive":>8s}  {"ToM":>5s}')
for label, chaos, reward, survive, tom in curriculum:
    print(f'{label:>8s}  {chaos:>5.1f}  {reward:>7.2f}  {survive:>8s}  {tom:>5s}')

## 5. Hackathon Theme Coverage

| Theme | How POLARIS Covers It |
|-------|---------------------|
| **Multi-Agent** | 5 LLM minister personas with negotiation, coalition formation, veto mechanics |
| **Long-Horizon** | Briefings with deadlines across 200+ steps, memory testing |
| **World Modeling** | 21-metric economic simulation with causal chains |
| **Self-Improvement** | Auto-curriculum that escalates difficulty as agent improves |

---

### Links
- **GitHub**: [POLARIS-V3](https://github.com/abhishekascodes/POLARIS-V3)
- **HF Space**: [asabhishek/polaris-v3](https://huggingface.co/spaces/asabhishek/polaris-v3)